In [6]:
# (Code skeleton written with help from Perplexity, and some manual adjustments)
#1. Load the data and process
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import classification_report

from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

mis = pd.read_csv("misuse.csv")

def discretize_score(x):
    if x <= 1.5:
        return 0  # low
    elif x <= 2.5:
        return 1  # medium
    else:
        return 2  # high

mis["conv_label"] = mis["Convincingness"].apply(discretize_score)
mis["seve_label"] = mis["Severity"].apply(discretize_score)

print(mis[["Review", "Convincingness", "conv_label", "Severity", "seve_label"]].head())

Device: cpu
                                              Review  Convincingness  \
0                              Great app 4 Stalking.             3.0   
1  I have this on my grandson’s phone.  It tells ...             4.0   
2  I read the negative review from a cry baby tee...             4.0   
3                     Good app to see your stalkers.             1.5   
4         Love it! Great to stalk family members lol             3.0   

   conv_label  Severity  seve_label  
0           2       3.0           2  
1           2       4.0           2  
2           2       4.0           2  
3           0       1.0           0  
4           2       3.0           2  


In [7]:
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

MODEL_NAMES = {
    "bert": "bert-base-uncased",
    "distilbert": "distilbert-base-uncased",
}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAMES["bert"])  # tokenizer 對兩個模型都通用

class MisuseDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=256):
        self.texts = df["Review"].astype(str).tolist()
        self.conv = df["conv_label"].astype(int).tolist()
        self.seve = df["seve_label"].astype(int).tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        enc = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["conv_label"] = torch.tensor(self.conv[idx], dtype=torch.long)
        item["seve_label"] = torch.tensor(self.seve[idx], dtype=torch.long)
        return item

Device: cpu


In [8]:
import torch.nn as nn
from transformers import AutoModel

class JointClassifier(nn.Module):
    def __init__(self, base_model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        hidden_size = self.encoder.config.hidden_size
        # two heads, 3 classes each
        self.conv_head = nn.Linear(hidden_size, 3)
        self.seve_head = nn.Linear(hidden_size, 3)

    def forward(self, input_ids, attention_mask=None, token_type_ids=None):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        # get [CLS] / pooled output
        if hasattr(outputs, "pooler_output") and outputs.pooler_output is not None:
            pooled = outputs.pooler_output
        else:
            pooled = outputs.last_hidden_state[:, 0, :]  # DistilBERT does not have pooler_output

        logits_conv = self.conv_head(pooled)
        logits_seve = self.seve_head(pooled)
        return logits_conv, logits_seve

In [9]:
from sklearn.metrics import classification_report
import torch
from torch.utils.data import DataLoader
import torch.nn as nn

def train_one_fold(model, train_ds, val_ds, epochs=2, batch_size=8, lr=2e-5):
    model.to(device)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for batch in tqdm(train_loader, desc=f"Train epoch {epoch+1}", leave=False):
            optimizer.zero_grad()

            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch.get("token_type_ids")
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(device)

            conv_label = batch["conv_label"].to(device)
            seve_label = batch["seve_label"].to(device)

            logits_conv, logits_seve = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )

            loss_conv = criterion(logits_conv, conv_label)
            loss_seve = criterion(logits_seve, seve_label)
            loss = loss_conv + loss_seve

            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_loss = total_loss / max(1, len(train_loader))
        print(f"  Train loss: {avg_loss:.4f}")

    # ---- evaluation ----
    model.eval()
    all_conv_true, all_conv_pred = [], []
    all_seve_true, all_seve_pred = [], []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Eval", leave=False):
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch.get("token_type_ids")
            if token_type_ids is not None:
                token_type_ids = token_type_ids.to(device)

            conv_true = batch["conv_label"].numpy()
            seve_true = batch["seve_label"].numpy()

            logits_conv, logits_seve = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
            )

            conv_pred = logits_conv.argmax(dim=-1).cpu().numpy()
            seve_pred = logits_seve.argmax(dim=-1).cpu().numpy()

            all_conv_true.extend(conv_true.tolist())
            all_conv_pred.extend(conv_pred.tolist())
            all_seve_true.extend(seve_true.tolist())
            all_seve_pred.extend(seve_pred.tolist())

    report_conv = classification_report(
        all_conv_true, all_conv_pred, output_dict=True, zero_division=0
    )
    report_seve = classification_report(
        all_seve_true, all_seve_pred, output_dict=True, zero_division=0
    )

    prec_conv = report_conv["macro avg"]["precision"]
    rec_conv  = report_conv["macro avg"]["recall"]
    f1_conv   = report_conv["macro avg"]["f1-score"]

    prec_seve = report_seve["macro avg"]["precision"]
    rec_seve  = report_seve["macro avg"]["recall"]
    f1_seve   = report_seve["macro avg"]["f1-score"]

    print("Convincingness macro P/R/F1:", prec_conv, rec_conv, f1_conv)
    print("Severity      macro P/R/F1:", prec_seve, rec_seve, f1_seve)

    return (prec_conv, rec_conv, f1_conv,
            prec_seve, rec_seve, f1_seve)

In [ ]:
from sklearn.model_selection import KFold
from statistics import mean

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def run_cv_for_model(base_name, label_name, epochs=2, batch_size=8, lr=2e-5):
    print(f"\n==== Model: {label_name} ({base_name}) ====")

    conv_P, conv_R, conv_F1 = [], [], []
    seve_P, seve_R, seve_F1 = [], [], []

    for fold, (train_idx, val_idx) in enumerate(kf.split(mis), 1):
        print(f"\n--- Fold {fold} ---")
        train_df = mis.iloc[train_idx].reset_index(drop=True)
        val_df   = mis.iloc[val_idx].reset_index(drop=True)

        train_ds = MisuseDataset(train_df, tokenizer)
        val_ds   = MisuseDataset(val_df, tokenizer)

        model = JointClassifier(base_model_name=base_name)
        (p_c, r_c, f1_c,
         p_s, r_s, f1_s) = train_one_fold(
            model,
            train_ds,
            val_ds,
            epochs=epochs,
            batch_size=batch_size,
            lr=lr,
        )

        conv_P.append(p_c); conv_R.append(r_c); conv_F1.append(f1_c)
        seve_P.append(p_s); seve_R.append(r_s); seve_F1.append(f1_s)

    summary = {
        "model": label_name,
        "conv_precision": mean(conv_P),
        "conv_recall": mean(conv_R),
        "conv_f1": mean(conv_F1),
        "seve_precision": mean(seve_P),
        "seve_recall": mean(seve_R),
        "seve_f1": mean(seve_F1),
    }
    print("\n[Summary]", summary)
    return summary

bert_summary = run_cv_for_model("bert-base-uncased", "BERT", epochs=2, batch_size=8, lr=2e-5)
distil_summary = run_cv_for_model("distilbert-base-uncased","DistilBERT", epochs=2, batch_size=8, lr=2e-5)

print("\n=== Final comparison ===")
for task in ["conv", "seve"]:
    # b_f1 = bert_summary[f"{task}_f1"]
    d_f1 = distil_summary[f"{task}_f1"]
    better = "BERT" if b_f1 > d_f1 else "DistilBERT"
    print(f"{task} macro F1 - BERT: {b_f1:.3f}, DistilBERT: {d_f1:.3f} → {better} slightly better")


==== Model: DistilBERT (distilbert-base-uncased) ====

--- Fold 1 ---


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6985.97it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Train loss: 1.0562


  Train loss: 0.6239


Convincingness macro P/R/F1: 0.57310643576738 0.634489861762589 0.5990716949854953
Severity      macro P/R/F1: 0.5692358130662852 0.6221132897603486 0.5892531876138434

--- Fold 2 ---


...

Convincingness macro P/R/F1: 0.5717783899602081 0.6022679037050512 0.5866270841244089
Severity      macro P/R/F1: 0.5747863247863249 0.5917222840192807 0.5830770617192164

--- Fold 5 ---


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 3195.90it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Train loss: 1.0488


  Train loss: 0.5959


Convincingness macro P/R/F1: 0.5613636363636364 0.5947799651503355 0.5775380533445049
Severity      macro P/R/F1: 0.554305769000207 0.5910404984423676 0.571864108964177

[Summary] {'model': 'DistilBERT', 'conv_precision': 0.5977535817008756, 'conv_recall': 0.6186772306822169, 'conv_f1': 0.6001760560056913, 'seve_precision': 0.5650185757728332, 'seve_recall': 0.602621593496126, 'seve_f1': 0.5806814187015098}

=== Final comparison ===
